# M1 · Student C — Matching-accuracy + speed protocol
**CSE 681 · Project 26 — Compact Deep Feature Descriptors vs. Classical SIFT/ORB**

| | |
|---|---|
| **Milestone** | M1 — Proposal & Literature Review (Week 3) |
| **Owner** | Student C (Evaluation & Integration Lead) |
| **Runtime** | CPU is fine · about 10 min |

**This notebook produces (saved to the shared Drive folder `DIP_P26/`):**
- `tables/m1_protocol.tex`: metric definitions (report §3/§4)
- unit tests showing that the metrics are computed correctly (keep the output as evidence)
- `results/m1_timing.csv` and `tables/m1_timing.tex`: **SIFT/ORB speed numbers** under the fixed timing protocol
- `figures/m1_protocol_example.png`: illustration of a correct vs. incorrect match

> First time? Read `README.md`: one teammate creates the shared folder `DIP_P26` on Google Drive,
> uploads `p26lib.py` into it and shares it; everyone adds a shortcut to *My Drive*.

In [ ]:
#@title 0 · Setup (run every session)
import os, sys, json, time
SMOKE_TEST = False   # True = 1-2 minute dry run on tiny SYNTHETIC data (to check the code). NEVER report smoke numbers.
SMOKE_TEST = SMOKE_TEST or os.environ.get("P26_SMOKE") == "1"

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJ, DATA = "/content/drive/MyDrive/DIP_P26", "/content/data"
except ImportError:                       # running locally / Kaggle
    PROJ = os.environ.get("P26_PROJ", os.path.abspath("DIP_P26"))
    DATA = os.environ.get("P26_DATA", os.path.abspath("data"))
os.makedirs(PROJ, exist_ok=True); os.makedirs(DATA, exist_ok=True)
sys.path.insert(0, PROJ)
assert os.path.exists(os.path.join(PROJ, "p26lib.py")), f"Upload p26lib.py into {PROJ} first (see README)."

import importlib, p26lib as P; importlib.reload(P)
import numpy as np, pandas as pd, cv2, torch
import matplotlib; import matplotlib.pyplot as plt
P.set_seed(P.SEED)
D = P.project_dirs(PROJ)
TAG = "SMOKE_" if SMOKE_TEST else ""          # smoke-test outputs never overwrite real ones
def OUT(kind, name):                           # e.g. OUT("tables", "m1_baseline.tex")
    return os.path.join(D[kind], TAG + name)
DEVICE = P.get_device()
print("SMOKE_TEST =", SMOKE_TEST, "| device =", DEVICE)
print(json.dumps(P.env_info(), indent=1))

In [ ]:
#@title 1 · HPatches sequences: uses the zip in Drive DIP_P26/data/ (copied + unzipped to the fast Colab disk, ~2 min per session)
if SMOKE_TEST:
    HP_ROOT = P.make_synthetic_hpatches(os.path.join(DATA, "synthetic_hpatches"), n_per_kind=2)
else:
    HP_ROOT = P.download_hpatches(DATA)
SEQS = P.list_sequences(HP_ROOT)          # D2-Net protocol: 108 sequences (52 illumination + 56 viewpoint)
print(len(SEQS), "sequences |", sum(P.seq_kind(s) == "illumination" for s in SEQS), "illumination,",
      sum(P.seq_kind(s) == "viewpoint" for s in SEQS), "viewpoint")

## 2 · The protocol (copy this into the report)
For every sequence, image 1 is matched to images 2–6, which gives 540 pairs on the 108 D2-Net sequences. Every method gets up to **2000 keypoints**.

| Metric | Definition |
|---|---|
| **MMA@t** | Mean Matching Accuracy: for each pair, the fraction of MNN matches $(p_1,p_2)$ with $\lVert \pi(H_{gt} p_1) - p_2 \rVert \le t$, averaged over pairs, for $t=1..10$ px (Dusmanu et al. 2019). |
| **H-acc@t** | Homography estimation accuracy: $\hat H$ from OpenCV RANSAC (3 px, 5000 iters, fixed seed). Correct when the mean distance of the 4 image corners mapped by $\hat H$ and by $H_{gt}$ is $\le t$, $t\in\{1,3,5\}$ (DeTone et al. 2018). |
| **#matches, inlier ratio** | MNN matches per pair; fraction kept by RANSAC. |
| **Extraction time** | Median of 10 runs after 2 warm-ups, detect + describe, **1 CPU thread** for every method (GPU reported separately for the CNN), on 10 fixed images. |
| **Descriptor-only time** | Same protocol, but only describing a fixed set of 2000 SIFT keypoints (isolates the descriptor cost). |
| **Memory** | Bytes per descriptor (SIFT 512 B float32, ORB 32 B, ours 4·d B or d/8 B binary). |

Matching is **mutual nearest neighbour**: L2 distance for float descriptors, Hamming for binary ones. MNN has no threshold to tune, so no method can be
tuned on the test set. Lowe's ratio test is reported as a sensitivity analysis at M2.

In [ ]:
#@title 2b · Protocol table -> tables/m1_protocol.tex
proto = pd.DataFrame([
    ["MMA@t", "fraction of MNN matches with GT reprojection error <= t px", "t = 1..10 px"],
    ["H-acc@t", "RANSAC homography, mean corner error <= t px", "t = 1, 3, 5 px; RANSAC 3 px"],
    ["matches / inlier ratio", "MNN matches per pair; RANSAC inliers / matches", "-"],
    ["extraction time", "median of 10 runs (2 warm-up), detect+describe, 1 CPU thread", "10 fixed images"],
    ["descriptor-only time", "describe 2000 fixed SIFT keypoints, same protocol", "10 fixed images"],
    ["memory", "bytes per descriptor", "-"],
], columns=["Metric", "Definition", "Settings"])
display(proto)
P.df_to_latex(proto, OUT("tables", "m1_protocol.tex"), "Evaluation protocol (HPatches, 108 sequences, 540 pairs, 2000 keypoints, MNN matching).", "tab:protocol")

## 3 · Unit tests: prove that the metrics are computed correctly

In [ ]:
#@title 3 · Metric unit tests (all must print OK)
rng = np.random.default_rng(0)
h, w = 480, 640
H = P.random_homography(h, w, rng)

# (1) warp_points is consistent with the inverse homography
pts = rng.uniform([0, 0], [w, h], (100, 2))
assert np.allclose(P.warp_points(P.warp_points(pts, H), np.linalg.inv(H)), pts, atol=1e-6); print("OK warp_points round-trip")

# (2) corner error: 0 for the exact homography, exactly 5 px for a 3-4-5 translation
T = np.array([[1, 0, 3], [0, 1, 4], [0, 0, 1.]])
assert P.corner_error(H, H, w, h) < 1e-9 and abs(P.corner_error(T, np.eye(3), w, h) - 5) < 1e-9; print("OK corner_error")

# (3) perfect correspondences -> MMA = 1 at every threshold, homography recovered
kp = [cv2.KeyPoint(float(x), float(y), 5) for x, y in pts]
kp2 = [cv2.KeyPoint(float(x), float(y), 5) for x, y in P.warp_points(pts, H)]
f1, f2 = P.Features(kp, np.eye(100, dtype=np.float32), cv2.NORM_L2), P.Features(kp2, np.eye(100, dtype=np.float32), cv2.NORM_L2)
m = P.match(f1, f2, "mnn")
assert len(m) == 100 and (m[:, 0] == m[:, 1]).all(); print("OK mutual-NN matcher finds the 100 identity matches")
r = P.evaluate_pair(f1, f2, m, H, (h, w))
assert r["mma@1"] == 1 and r["hacc@1"] == 1 and r["corner_err"] < 1e-3; print("OK perfect matches -> MMA=1, H-acc=1")

# (4) shuffled correspondences -> MMA ~ 0, homography fails
r2 = P.evaluate_pair(f1, f2, np.stack([m[:, 0], rng.permutation(m[:, 1])], 1), H, (h, w))
assert r2["mma@10"] < 0.1 and r2["hacc@5"] == 0; print("OK random matches -> MMA≈0, H-acc=0")

# (5) 2 px Gaussian noise -> MMA@1 < MMA@3 < MMA@10 (monotone in t)
kp3 = [cv2.KeyPoint(float(x), float(y), 5) for x, y in P.warp_points(pts, H) + rng.normal(0, 2, (100, 2))]
r3 = P.evaluate_pair(f1, P.Features(kp3, f2.desc, cv2.NORM_L2), m, H, (h, w))
assert r3["mma@1"] <= r3["mma@3"] <= r3["mma@10"] == 1; print("OK MMA monotone:", r3["mma@1"], r3["mma@3"], r3["mma@10"])

# (6) FPR95: perfectly separable -> 0, identical distributions -> ~0.95
assert P.fpr95(np.r_[rng.uniform(0, 1, 1000), rng.uniform(2, 3, 1000)], np.r_[np.ones(1000), np.zeros(1000)]) == 0
x = rng.uniform(0, 1, 20000); assert abs(P.fpr95(x, np.arange(20000) % 2) - 0.95) < 0.02; print("OK FPR95")

# (7) Hamming matcher for binary descriptors
b = np.packbits(rng.integers(0, 2, (50, 256)).astype(np.uint8), axis=1)
fb = P.Features(kp[:50], b, cv2.NORM_HAMMING)
assert (P.match(fb, fb)[:, 1] == np.arange(50)).all(); print("OK Hamming MNN")
print("\nAll protocol unit tests passed.")

In [ ]:
#@title 3b · Illustration of a correct vs. incorrect match -> figures/m1_protocol_example.png
sd = [s for s in SEQS if P.seq_kind(s) == "viewpoint"][0]
imgs, Hs = P.load_sequence(sd)
e = P.SIFTExtractor(2000); f1, f2 = e(imgs[0]), e(imgs[3]); m = P.match(f1, f2)
err = np.linalg.norm(P.warp_points(f1.pts[m[:, 0]], Hs[4]) - f2.pts[m[:, 1]], axis=1)
good, bad = m[err <= 3], m[err > 3]
vis = np.hstack([cv2.cvtColor(imgs[0], cv2.COLOR_GRAY2BGR), cv2.cvtColor(imgs[3], cv2.COLOR_GRAY2BGR)])
off = imgs[0].shape[1]
for mm, col in [(good[:80], (0, 200, 0)), (bad[:40], (0, 0, 255))]:
    for i, j in mm:
        p1 = tuple(int(v) for v in f1.pts[i]); p2 = tuple(int(v) for v in f2.pts[j]); p2 = (p2[0] + off, p2[1])
        cv2.line(vis, p1, p2, col, 1, cv2.LINE_AA)
plt.figure(figsize=(15, 6)); plt.imshow(vis[..., ::-1]); plt.axis("off")
plt.title(f"SIFT {os.path.basename(sd)} 1-4: green = correct at 3 px ({len(good)}), red = wrong ({len(bad)}); MMA@3 = {len(good)/max(len(m),1):.2f}")
plt.savefig(OUT("figures", "m1_protocol_example.png"), dpi=120, bbox_inches="tight"); plt.show()

## 4 · Speed protocol + baseline speed numbers

In [ ]:
#@title 4 · Timing (1 CPU thread, median of 10 after 2 warm-ups, 10 fixed images) -> tables/m1_timing.tex
P.set_threads(1)
rng = np.random.default_rng(P.SEED)
N_IMG = 2 if SMOKE_TEST else 10
timing_imgs = [P.load_sequence(SEQS[k])[0][0] for k in sorted(rng.choice(len(SEQS), N_IMG, replace=False))]
REPEATS = 3 if SMOKE_TEST else 10
rows = []
for e in [P.SIFTExtractor(2000), P.ORBExtractor(2000)]:
    t = [P.time_extractor(e, im, repeats=REPEATS)["extract_ms_median"] for im in timing_imgs]
    sift_kps = [cv2.SIFT_create(2000).detect(im, None) for im in timing_imgs]
    desc_only = [P.time_describe_only(e, im, k, repeats=REPEATS) for im, k in zip(timing_imgs, sift_kps)] if e.name == "SIFT" else [np.nan]
    rows.append({"method": e.name, "extract ms (median over imgs)": np.median(t), "describe-only ms": np.median(desc_only) if e.name == "SIFT" else np.nan,
                 "B/desc": e.bytes_per_desc, "dim": e.dim})
P.set_threads(os.cpu_count())
timing = pd.DataFrame(rows)
timing.to_csv(OUT("results", "m1_timing.csv"), index=False)
display(timing.round(1))
P.df_to_latex(timing.round(1), OUT("tables", "m1_timing.tex"), "Baseline speed (1 CPU thread, 2000 keypoints, median of 10 runs on 10 fixed HPatches images).", "tab:m1_timing", fmt="{:.1f}")

In [ ]:
#@title 5 · Cross-check Student A's baseline results (if A already ran M1_A)
pth = OUT("results", "m1_baseline_pairs.csv")
if os.path.exists(pth):
    dfa = pd.read_csv(pth)
    assert dfa.groupby("method").size().nunique() == 1, "methods were not evaluated on the same pairs!"
    assert set(dfa["pair"]) == {f"1-{i}" for i in range(2, 7)}
    assert ((dfa["mma@1"] <= dfa["mma@3"] + 1e-9) & (dfa["mma@3"] <= dfa["mma@10"] + 1e-9)).all()
    print("Student A's results are consistent with the protocol:", dfa.groupby("method").size().to_dict(), "pairs per method")
    display(P.summarize(dfa)[["mma@3", "hacc@3", "n_matches"]].round(3))
else:
    print("Run M1_A first, then re-run this cell.")

## ✅ Deliverable checklist & where it goes in the report

- [ ] Protocol text and `tables/m1_protocol.tex` go in **§3** (or a short "Evaluation protocol" paragraph in §4)
- [ ] `tables/m1_timing.tex` sits next to Student A's accuracy table: together they are the **"SIFT/ORB baseline numbers"** deliverable
- [ ] Keep the unit-test output (save the executed notebook to GitHub). It shows the metrics are correct, which you will need at M2 for "experimental rigor".
- [ ] Assemble the M1 PDF from the template: Abstract, §1–§7, `references.bib`, and the **work-breakdown table** (copy from the catalog with your names)
- [ ] Add the AI-use disclosure line if you used AI tools